# memopro in 5 minutes

memopro gives back memory held by models you are not using right now, and shows where training
memory goes. This notebook runs anywhere in a few seconds: it builds a small GPT-2 locally.
Swap in your own model (for example `GPT2LMHeadModel.from_pretrained("gpt2")`) to see real sizes.

In [ ]:
%load_ext memopro
import memopro, torch
from transformers import GPT2Config, GPT2LMHeadModel

print(memopro.doctor(devices=False).summary())

## 1. Two models, one notebook

`model_a` is loaded from a safetensors file, as `from_pretrained` does; `model_b` is a changed copy
(as after fine-tuning), so it no longer matches any file.

In [ ]:
import tempfile
workdir = tempfile.mkdtemp()
torch.manual_seed(0)
GPT2LMHeadModel(GPT2Config(n_layer=4, n_embd=256, n_head=4, vocab_size=8000)).save_pretrained(workdir)

model_a = GPT2LMHeadModel.from_pretrained(workdir).eval()
model_b = GPT2LMHeadModel.from_pretrained(workdir).eval()
with torch.no_grad():
    for p in model_b.parameters():
        p.mul_(1.001)

x = torch.randint(0, 8000, (1, 32))
with torch.no_grad():
    reference_a = model_a(x).logits.clone()

## 2. Compare the methods before choosing

Nothing changes yet. `source` re-reads the original file (no SSD write); `spill` would write to the
SSD and needs your consent; `bf16` changes numerics and is never chosen automatically.

In [ ]:
%hibernate model_a --plan

In [ ]:
%hibernate model_a
%hibernate model_b
%memopro status

## 3. Use it again: it wakes up by itself

Calling a hibernated model restores it first. The result is bit-for-bit the same.

In [ ]:
with torch.no_grad():
    out = model_a(x).logits
print("identical to before:", torch.equal(out, reference_a))

## 4. Where does training memory go?

`census` records one training step: bytes per category, how compressible they are, how many bits
they need, and what you could do about it.

In [ ]:
optimizer = torch.optim.AdamW(model_b.parameters())
batch = torch.randint(0, 8000, (4, 64))
with memopro.census.record(model_b, optimizer, mode="light") as c:
    loss = model_b(batch, labels=batch).loss
    loss.backward()
    optimizer.step()
print(c.summary())

## What memopro did not do

- It wrote nothing to your SSD (`disk_writes="ask"` by default).
- It did not change any numbers: `source` and `compress` restore exactly.
- Settings: `memopro.configure(...)`, a `memopro.toml` file, or `MEMOPRO_*` environment variables.